# DLRM — Deep Learning Recommendation Model
**Stage: Re-Ranking** — fine-grained scoring of candidates from Two-Tower retrieval.

## Architecture
```
Dense features  ──► Bottom MLP ─────────────────────────────────┐
(13 continuous)     [in → 64 → 16]                              │
                                                                 ├──► Feature Interaction ──► Top MLP ──► P(click)
Sparse features ──► Embedding Tables ──► [e1, e2, …, e6]       │
(6 categorical)     user_id, item_id, category,                 │
                    brand, country, device                       │
```
**Paper:** Naumov et al. 2019 · **Used by:** Meta, Twitter, Alibaba

## Free GPU options
- **Google Colab** → Runtime → Change runtime type → **T4 GPU**
- **Kaggle** → Settings → Accelerator → GPU (30 hrs/week free)

## Cell 0 — Install packages
Run once, then **Runtime → Restart session**, then run all cells from Cell 1.

In [ ]:
import sys
pkgs = [
    'torch==2.11.0',
    'mlflow==3.11.1',
    'scikit-learn==1.8.0',
    'pandas==2.2.3',
    'numpy==1.26.4',
    'matplotlib',
    'tqdm',
]
!{sys.executable} -m pip install --quiet {' '.join(pkgs)}
print('Done. Now go to Runtime → Restart session, then run from Cell 1.')

## Cell 1 — Setup: clone repo & set path
Run this first after every runtime restart.

In [ ]:
import os, sys, shutil

REPO_URL = 'https://github.com/mutellipiqbal/dlrm-recommender'
REPO_DIR = '/content/dlrm-recommender'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print(f'Working dir : {os.getcwd()}')
print(f'src/ exists : {os.path.exists("src")}')
print(f'Files       : {os.listdir(".")}')

## Cell 2 — Imports & config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mlflow
from tqdm.auto import tqdm

from src.model   import DLRM
from src.dataset import (
    generate_synthetic_data, split_data, make_loaders,
    NUM_DENSE, NUM_SPARSE, SPARSE_KEYS, VOCAB_SIZES,
)
from src.trainer import train, evaluate

CFG = {
    'embedding_dim':     16,
    'bottom_mlp_dims':   [64, 16],
    'top_mlp_dims':      [256, 128, 64],
    'dropout':           0.1,
    'n_samples':         500_000,
    'pos_weight':        5.0,
    'batch_size':        4096,
    'lr':                1e-3,
    'weight_decay':      1e-5,
    'epochs':            10,
    'seed':              42,
    'mlflow_experiment': 'dlrm_reranker',
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 3 — Generate synthetic data
Criteo-style: 13 dense + 6 sparse features, 500K rows.

In [ ]:
df = generate_synthetic_data(n_samples=CFG['n_samples'], seed=CFG['seed'])

print(f'\nFeature schema:')
print(f'  Dense  ({NUM_DENSE}): dense_0 … dense_{NUM_DENSE - 1}')
print(f'  Sparse ({NUM_SPARSE}): {SPARSE_KEYS}')
print(f'\nVocab sizes:')
for k, v in VOCAB_SIZES.items():
    print(f'  {k:<20}: {v:,}')

df.head()

## Cell 4 — Split & build DataLoaders

In [ ]:
train_df, val_df, test_df = split_data(df, seed=CFG['seed'])
train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, batch_size=CFG['batch_size']
)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## Cell 5 — Instantiate DLRM
Bottom MLP: 13 dense → 64 → **16** (= `embedding_dim`)
6 sparse tables each → 16-dim vector
Feature interaction: dot-product of 7 vectors (1 dense + 6 sparse) → 21 unique pairs
Top MLP input: 16 + 21 = **37** → 256 → 128 → 64 → 1 logit

In [ ]:
model = DLRM(
    num_dense_features = NUM_DENSE,
    vocab_sizes        = list(VOCAB_SIZES.values()),
    embedding_dim      = CFG['embedding_dim'],
    bottom_mlp_dims    = CFG['bottom_mlp_dims'],
    top_mlp_dims       = CFG['top_mlp_dims'],
    dropout            = CFG['dropout'],
).to(DEVICE)

print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

# Sanity-check forward pass
dummy_dense  = torch.randn(8, NUM_DENSE).to(DEVICE)
dummy_sparse = torch.stack(
    [torch.randint(0, v, (8,)) for v in VOCAB_SIZES.values()], dim=1
).to(DEVICE)

out = model(dummy_dense, dummy_sparse)
print(f'Output shape: {out.shape}  ← should be torch.Size([8])')

## Cell 6 — Train (MLflow tracked)
- **AdamW** + **OneCycleLR** (5% warmup + cosine decay)
- **BCEWithLogitsLoss** with `pos_weight=5.0` for class imbalance
- **Early stopping** after 3 epochs without val AUC improvement

In [ ]:
model = train(
    model, train_loader, val_loader,
    cfg=CFG, device=DEVICE, run_name='dlrm_v1'
)

## Cell 7 — Test evaluation

In [ ]:
pos_weight   = torch.tensor([CFG['pos_weight']], device=DEVICE)
loss_fn      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
test_metrics = evaluate(model, test_loader, loss_fn, DEVICE)

print(f'Test AUC   : {test_metrics["auc"]:.4f}   (primary ranking metric)')
print(f'Test PR-AUC: {test_metrics["prauc"]:.4f}  (important for imbalanced labels)')
print(f'Test Loss  : {test_metrics["loss"]:.4f}')

## Cell 8 — Feature importance: embedding norms
Sparse features with higher mean embedding norm have learned more signal.
This is the standard interpretability proxy used in production DLRM systems.

In [ ]:
# Unwrap torch.compile if active
m = model._orig_mod if hasattr(model, '_orig_mod') else model

# sparse_embeddings is nn.ModuleList — access tables directly by index
norms = [
    m.sparse_embeddings[i].weight.data.norm(dim=1).mean().item()
    for i in range(NUM_SPARSE)
]

plt.figure(figsize=(8, 4))
plt.bar(SPARSE_KEYS, norms, color='steelblue', edgecolor='white')
plt.title('Mean Embedding Norm per Sparse Feature\n(higher = more signal learned)')
plt.ylabel('Mean L2 Norm')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('embedding_norms.png', dpi=120)
plt.show()
print('Saved embedding_norms.png')

## Cell 9 — Score new candidates (inference)
In production these 100 candidates come from the Two-Tower FAISS retrieval step.

In [ ]:
model.eval()
n_candidates = 100

dense_cands  = torch.randn(n_candidates, NUM_DENSE).to(DEVICE)
sparse_cands = torch.stack(
    [torch.randint(0, v, (n_candidates,)) for v in VOCAB_SIZES.values()], dim=1
).to(DEVICE)

with torch.no_grad():
    scores = torch.sigmoid(model(dense_cands, sparse_cands)).cpu().numpy()

top10_idx = scores.argsort()[::-1][:10]
print('Top-10 candidate indices (re-ranked by DLRM):')
for rank, idx in enumerate(top10_idx, 1):
    print(f'  {rank:>2}. Candidate {idx:>3}  score={scores[idx]:.4f}')

## Cell 10 — MLflow run summary

In [ ]:
runs = mlflow.search_runs(experiment_names=[CFG['mlflow_experiment']])
cols = ['run_id', 'metrics.val_auc', 'metrics.val_prauc', 'metrics.best_val_auc']
print(runs[cols].sort_values('metrics.val_auc', ascending=False).head())

## Cell 11 — Save model
Save weights locally and optionally push to Hugging Face Hub.

In [ ]:
import json

# Save weights
torch.save(model.state_dict(), 'best_dlrm.pt')
print('Saved best_dlrm.pt')

# Save config (needed by deploy/app.py)
meta = {
    'num_dense_features': NUM_DENSE,
    'vocab_sizes':        list(VOCAB_SIZES.values()),
    'embedding_dim':      CFG['embedding_dim'],
    'bottom_mlp_dims':    CFG['bottom_mlp_dims'],
    'top_mlp_dims':       CFG['top_mlp_dims'],
    'dropout':            CFG['dropout'],
    'sparse_feature_names': SPARSE_KEYS,
}
with open('dlrm_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved dlrm_meta.json')
print('\nNext: run deploy/train_and_push.py to push to Hugging Face Hub')

---
## Pipeline summary
```
All Items (50K)
      │
      ▼
[Two-Tower] ── FAISS ANN ──► top-100 candidates   (fast, coarse)
      │
      ▼
[DLRM]      ── score each ──► top-10 final items  (precise, fine-grained)
```
See companion project: **`two-tower-recommender`**